# 🧠 XGBoost — Predicción de Nivel de Estrés en Adolescentes
### Dataset: Teen Mental Health (Redes Sociales, Sueño y Bienestar)

In [ ]:
# Instalar si es necesario
# pip install xgboost scikit-learn pandas numpy matplotlib seaborn openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import xgboost as xgb
from xgboost import XGBRegressor, plot_importance

import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías cargadas correctamente')
print(f'XGBoost version: {xgb.__version__}')

In [ ]:
# ── Cargar el Excel ──────────────────────────────────────────────────────────
df = pd.read_excel('datosestresyansiedad.xlsx')

print('=' * 55)
print('📊 INFORMACIÓN DEL DATASET')
print('=' * 55)
print(f'Shape: {df.shape}')
print(f'\nColumnas: {list(df.columns)}')
print(f'\nValores nulos:\n{df.isnull().sum()}')
print(f'\nEstadísticas descriptivas:')
df.describe().round(2)

In [ ]:
# ── Codificación de variables categóricas ───────────────────────────────────
# gender: male=1, female=0
# platform_usage: Instagram=0, TikTok=1, Both=2
# social_interaction_level: low=0, medium=1, high=2

df_encoded = df.copy()

df_encoded['gender'] = df_encoded['gender'].map({'male': 1, 'female': 0})
df_encoded['platform_usage'] = df_encoded['platform_usage'].map({'Instagram': 0, 'TikTok': 1, 'Both': 2})
df_encoded['social_interaction_level'] = df_encoded['social_interaction_level'].map({'low': 0, 'medium': 1, 'high': 2})

print('✅ Codificación completada')
print(f'\nDistribución del target (stress_level):')
print(df_encoded['stress_level'].value_counts().sort_index())
print(f'\nRango: {df_encoded["stress_level"].min()} — {df_encoded["stress_level"].max()}')
print(f'Media: {df_encoded["stress_level"].mean():.2f}')

In [ ]:
# ── Visualizar distribución de la variable objetivo ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_encoded['stress_level'], bins=10, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribución de Stress Level', fontsize=13)
axes[0].set_xlabel('Nivel de Estrés (1–10)')
axes[0].set_ylabel('Frecuencia')

axes[1].boxplot(df_encoded['stress_level'], patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_title('Boxplot de Stress Level', fontsize=13)
axes[1].set_ylabel('Nivel de Estrés')

plt.tight_layout()
plt.show()

In [ ]:
# ── Mapa de correlación ──────────────────────────────────────────────────────
plt.figure(figsize=(13, 9))
corr_matrix = df_encoded.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5,
    annot_kws={'size': 8}
)
plt.title('Mapa de Correlación — Teen Mental Health Dataset', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Separar features y target ────────────────────────────────────────────────
# Excluimos depression_label (etiqueta extra) y la variable objetivo
X = df_encoded.drop(columns=['stress_level', 'depression_label'])
y = df_encoded['stress_level']

feature_names = X.columns.tolist()
print(f'Features ({len(feature_names)}): {feature_names}')
print(f'Target: stress_level | Shape: {y.shape}')

# ── Train / Test Split ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f'\n📐 Tamaños:')
print(f'  Train: {X_train.shape[0]} muestras')
print(f'  Test : {X_test.shape[0]} muestras')

# ── Escalar (opcional para XGBoost, útil para comparaciones) ────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_names)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=feature_names)

In [ ]:
# ── Modelo con parámetros por defecto ────────────────────────────────────────
xgb_base = XGBRegressor(
    objective    = 'reg:squarederror',
    random_state = 42,
    n_jobs       = -1,
    verbosity    = 0
)

xgb_base.fit(X_train, y_train)
y_pred_base = xgb_base.predict(X_test)

# ── Métricas ─────────────────────────────────────────────────────────────────
def evaluar_modelo(y_true, y_pred, nombre='Modelo'):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    # Evitar división por cero en MAPE
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

    print(f"\n{'='*45}")
    print(f'  📈 Resultados — {nombre}')
    print(f"{'='*45}")
    print(f'  RMSE : {rmse:.4f}')
    print(f'  MAE  : {mae:.4f}')
    print(f'  R²   : {r2:.4f}')
    print(f'  MAPE : {mape:.2f}%')
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': mape}

metricas_base = evaluar_modelo(y_test, y_pred_base, 'XGBoost Base')

In [ ]:
# ── Grid de parámetros ───────────────────────────────────────────────────────
param_grid = {
    'n_estimators'    : [100, 200, 300],
    'max_depth'       : [3, 5, 7],
    'learning_rate'   : [0.05, 0.1, 0.2],
    'subsample'       : [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3]
}

xgb_cv = XGBRegressor(
    objective    = 'reg:squarederror',
    random_state = 42,
    n_jobs       = -1,
    verbosity    = 0
)

grid_search = GridSearchCV(
    estimator  = xgb_cv,
    param_grid = param_grid,
    cv         = 5,
    scoring    = 'neg_root_mean_squared_error',
    n_jobs     = -1,
    verbose    = 1
)

print('🔎 Ejecutando GridSearchCV... (puede tardar unos minutos)')
grid_search.fit(X_train, y_train)

print(f'\n✅ Mejores parámetros encontrados:')
for k, v in grid_search.best_params_.items():
    print(f'   {k:20s}: {v}')
print(f'\n   CV RMSE (mejor): {-grid_search.best_score_:.4f}')

## Modelo Optimizado

In [ ]:
# ── Entrenar con mejores hiperparámetros ─────────────────────────────────────
best_params = grid_search.best_params_

xgb_opt = XGBRegressor(
    **best_params,
    objective    = 'reg:squarederror',
    random_state = 42,
    n_jobs       = -1,
    verbosity    = 0,
    early_stopping_rounds = 20
)

xgb_opt.fit(
    X_train, y_train,
    eval_set = [(X_train, y_train), (X_test, y_test)],
    verbose  = False
)

y_pred_opt = xgb_opt.predict(X_test)
metricas_opt = evaluar_modelo(y_test, y_pred_opt, 'XGBoost Optimizado')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('XGBoost — Teen Mental Health: Predicción de Nivel de Estrés', fontsize=16, fontweight='bold')

# ── 1. Predicciones vs Valores Reales ────────────────────────────────────────
ax = axes[0, 0]
ax.scatter(y_test, y_pred_opt, alpha=0.6, color='steelblue', edgecolors='white', s=60)
lims = [min(y_test.min(), y_pred_opt.min()) - 0.5,
        max(y_test.max(), y_pred_opt.max()) + 0.5]
ax.plot(lims, lims, 'r--', linewidth=2, label='Predicción perfecta')
ax.set_xlabel('Valores Reales')
ax.set_ylabel('Predicciones')
ax.set_title('Predicciones vs Valores Reales')
ax.legend()
ax.text(0.05, 0.92, f'R² = {metricas_opt["R2"]:.4f}',
        transform=ax.transAxes, fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# ── 2. Residuos ───────────────────────────────────────────────────────────────
ax = axes[0, 1]
residuos = y_test.values - y_pred_opt
ax.scatter(y_pred_opt, residuos, alpha=0.6, color='coral', edgecolors='white', s=60)
ax.axhline(y=0, color='black', linewidth=1.5, linestyle='--')
ax.set_xlabel('Predicciones')
ax.set_ylabel('Residuos')
ax.set_title('Gráfico de Residuos')

# ── 3. Distribución de Residuos ───────────────────────────────────────────────
ax = axes[0, 2]
ax.hist(residuos, bins=25, color='mediumpurple', edgecolor='white', alpha=0.8)
ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5)
ax.set_xlabel('Residuos')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de Residuos')
ax.text(0.05, 0.92, f'μ={residuos.mean():.2f}\nσ={residuos.std():.2f}',
        transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lavender', alpha=0.7))

# ── 4. Feature Importance ─────────────────────────────────────────────────────
ax = axes[1, 0]
importances = xgb_opt.feature_importances_
indices     = np.argsort(importances)[::-1]
sorted_feat = [feature_names[i] for i in indices]
sorted_imp  = importances[indices]

bars = ax.barh(sorted_feat[::-1], sorted_imp[::-1],
               color=plt.cm.viridis(np.linspace(0.2, 0.9, len(sorted_feat))))
ax.set_xlabel('Importancia (F-score)')
ax.set_title('Importancia de Variables')
for bar, val in zip(bars, sorted_imp[::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8)

# ── 5. Curva de Aprendizaje ───────────────────────────────────────────────────
ax = axes[1, 1]
results = xgb_opt.evals_result()
epochs  = len(results['validation_0']['rmse'])
x_axis  = range(epochs)

ax.plot(x_axis, results['validation_0']['rmse'],
        label='Train RMSE', color='steelblue', linewidth=2)
ax.plot(x_axis, results['validation_1']['rmse'],
        label='Test RMSE',  color='coral',     linewidth=2)
ax.set_xlabel('Número de Árboles')
ax.set_ylabel('RMSE')
ax.set_title('Curva de Aprendizaje')
ax.legend()
ax.grid(True, alpha=0.3)

# ── 6. Comparación Base vs Optimizado ────────────────────────────────────────
ax = axes[1, 2]
metricas_labels = ['RMSE', 'MAE', 'R²', 'MAPE (%)']
base_vals = [metricas_base['RMSE'], metricas_base['MAE'],
             metricas_base['R2'],   metricas_base['MAPE']]
opt_vals  = [metricas_opt['RMSE'],  metricas_opt['MAE'],
             metricas_opt['R2'],    metricas_opt['MAPE']]

x_pos = np.arange(len(metricas_labels))
width = 0.35
ax.bar(x_pos - width/2, base_vals, width, label='Base',       color='steelblue', alpha=0.8)
ax.bar(x_pos + width/2, opt_vals,  width, label='Optimizado', color='coral',     alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(metricas_labels)
ax.set_title('Comparación: Base vs Optimizado')
ax.legend()
for i, (b, o) in enumerate(zip(base_vals, opt_vals)):
    ax.text(i - width/2, b + 0.02, f'{b:.2f}', ha='center', fontsize=8)
    ax.text(i + width/2, o + 0.02, f'{o:.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('xgboost_estres_resultados.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figura guardada como 'xgboost_estres_resultados.png'")

## Validación Cruzada Final

In [ ]:
import copy

# ── Cross-Validation con el modelo optimizado ────────────────────────────────
xgb_opt_cv = copy.deepcopy(xgb_opt)
xgb_opt_cv.set_params(early_stopping_rounds=None)

cv_scores = cross_val_score(
    xgb_opt_cv, X, y,
    cv      = 10,
    scoring = 'neg_root_mean_squared_error',
    n_jobs  = -1
)

cv_rmse = -cv_scores

print('\n' + '='*50)
print('  🔁 Validación Cruzada (10-Fold)')
print('='*50)
print(f'  RMSE por fold: {np.round(cv_rmse, 3)}')
print(f'  RMSE Media   : {cv_rmse.mean():.4f}')
print(f'  RMSE Std     : {cv_rmse.std():.4f}')
print(f'  IC 95%       : [{cv_rmse.mean()-2*cv_rmse.std():.4f}, '
                        f'{cv_rmse.mean()+2*cv_rmse.std():.4f}]')

# ── Tabla resumen final ───────────────────────────────────────────────────────
print('\n' + '='*50)
print('  📊 RESUMEN FINAL')
print('='*50)
resumen = pd.DataFrame({
    'Métrica'   : ['RMSE', 'MAE', 'R²', 'MAPE (%)'],
    'Base'      : [round(metricas_base[k], 4) for k in ['RMSE','MAE','R2','MAPE']],
    'Optimizado': [round(metricas_opt[k], 4)  for k in ['RMSE','MAE','R2','MAPE']]
})
resumen['Mejora (%)'] = (
    (resumen['Base'] - resumen['Optimizado']) / resumen['Base'] * 100
).round(2)
print(resumen.to_string(index=False))

## Predicción

In [ ]:
# ── Un solo registro como diccionario ────────────────────────────────────────
# Codificación: gender (male=1, female=0) | platform (Instagram=0, TikTok=1, Both=2)
#               social_interaction_level (low=0, medium=1, high=2)

nuevo_registro = {
    'age'                     : 16,    # Edad del adolescente
    'gender'                  : 1,     # 1=male, 0=female
    'daily_social_media_hours': 5.0,   # Horas diarias en redes sociales
    'platform_usage'          : 2,     # 0=Instagram, 1=TikTok, 2=Both
    'sleep_hours'             : 6.5,   # Horas de sueño
    'screen_time_before_sleep': 2.0,   # Horas de pantalla antes de dormir
    'academic_performance'    : 3.0,   # Rendimiento académico (escala 0–4)
    'physical_activity'       : 1.0,   # Horas de actividad física
    'social_interaction_level': 1,     # 0=low, 1=medium, 2=high
    'anxiety_level'           : 5,     # Nivel de ansiedad (1–10)
    'addiction_level'         : 6      # Nivel de adicción (1–10)
}

# ── Convertir a DataFrame ─────────────────────────────────────────────────────
df_nuevo = pd.DataFrame([nuevo_registro])

# ── Predecir ──────────────────────────────────────────────────────────────────
prediccion = xgb_opt.predict(df_nuevo)[0]

print('=' * 45)
print('  🧠 PREDICCIÓN — Registro Individual')
print('=' * 45)
print(f'\n  Nivel de estrés estimado: {prediccion:.2f} / 10')
nivel = 'Bajo' if prediccion <= 3 else ('Medio' if prediccion <= 6 else 'Alto')
print(f'  Categoría               : {nivel}')

## Predicción con Múltiples Registros

In [ ]:
# ── Varios perfiles de adolescentes ──────────────────────────────────────────
nuevos_registros = pd.DataFrame({
    'age'                     : [14,   17,   15,   18,   16  ],
    'gender'                  : [0,    1,    0,    1,    0   ],
    'daily_social_media_hours': [1.5,  7.5,  4.0,  6.0,  2.5 ],
    'platform_usage'          : [0,    1,    2,    1,    0   ],
    'sleep_hours'             : [8.5,  5.0,  6.5,  4.5,  7.5 ],
    'screen_time_before_sleep': [0.5,  3.0,  2.0,  2.8,  1.0 ],
    'academic_performance'    : [3.8,  2.1,  3.0,  2.5,  3.5 ],
    'physical_activity'       : [2.0,  0.3,  1.0,  0.5,  1.8 ],
    'social_interaction_level': [2,    0,    1,    0,    2   ],
    'anxiety_level'           : [2,    8,    5,    9,    3   ],
    'addiction_level'         : [1,    9,    5,    8,    2   ]
})

# ── Etiquetas descriptivas ────────────────────────────────────────────────────
etiquetas = [
    'Adolescente saludable',
    'Alto uso redes + poco sueño',
    'Perfil promedio',
    'Alto estrés académico + TikTok',
    'Bajo uso redes + activo'
]

# ── Predecir todos ────────────────────────────────────────────────────────────
predicciones = xgb_opt.predict(nuevos_registros)

# ── Mostrar resultados ────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('  🧠 PREDICCIONES — Múltiples Perfiles')
print('=' * 60)
for etiqueta, pred in zip(etiquetas, predicciones):
    nivel = 'Bajo' if pred <= 3 else ('Medio' if pred <= 6 else 'Alto')
    print(f'  {etiqueta:<35} → Estrés: {pred:.2f} [{nivel}]')

# ── Gráfico de barras de predicciones ────────────────────────────────────────
plt.figure(figsize=(10, 4))
colores = ['green' if p <= 3 else ('orange' if p <= 6 else 'red') for p in predicciones]
bars = plt.barh(etiquetas, predicciones, color=colores, alpha=0.75, edgecolor='white')
plt.axvline(x=3, color='green',  linestyle='--', linewidth=1, label='Umbral Bajo')
plt.axvline(x=6, color='orange', linestyle='--', linewidth=1, label='Umbral Medio')
plt.xlabel('Nivel de Estrés Predicho (1–10)')
plt.title('Predicción de Estrés por Perfil', fontsize=13)
plt.xlim(0, 10)
plt.legend()
for bar, val in zip(bars, predicciones):
    plt.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
             f'{val:.2f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()